In [ ]:
import torch
import gpytorch
import pandas as pd
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from torch.utils.data import TensorDataset, DataLoader
from pyproj import Transformer
from sklearn.metrics import pairwise_distances
from scipy.interpolate import RegularGridInterpolator
from torch_geometric.data import Data




In [ ]:
#compare the graph vs no graph model on 10 seeds. 

In [ ]:
# ================================================================
# 10-SEED COMPARISON: wind_graph (region-mean wind) vs no_graph
# Continuous 36h trajectory, same train/val/test split (2016-17/2018/2019)
# no_graph = identical architecture, edge weights forced to zero
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)  # (B, n_reg, HORIZON) logits

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

# ---- wind edges ----
wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight (per-hour trajectory): {POS_WEIGHT:.2f}")

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)
del station_wind_edge_weight, region_wind_edge_weight
gc.collect()

# no_graph: identical topology, edge weights forced to zero -> WindConvLayer's
# neighbor/connectivity terms vanish, leaving only the self-transform.
zero_ewt_s = torch.zeros_like(wind_ewt_s)
zero_ewt_r = torch.zeros_like(wind_ewt_r)

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def train_one_seed_final(seed, ewt_s, ewt_r, lr=3e-3, dropout=0.5, weight_decay=5e-4):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch(p1, ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch(p1, ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_conv)
    p2 = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_traj_epoch(p2, ewt_s, ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
        run_p2_traj_epoch(p2, ewt_s, ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
        val_probs = predict_traj_probs(p2, ewt_s, ewt_r, X1_t, starts1_t)
        val_agg_score = val_probs.max(axis=-1).reshape(-1)
        va = average_precision_score(y_val_agg, val_agg_score)
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

# ================================================================
# RUN: 10 seeds x {wind_graph, no_graph}
# ================================================================
N_SEEDS = 10
results = {"wind_graph": [], "no_graph": []}

for seed in range(N_SEEDS):
    for model_name, ewt_s, ewt_r in [("wind_graph", wind_ewt_s, wind_ewt_r), ("no_graph", zero_ewt_s, zero_ewt_r)]:
        print(f"\n{'='*10} seed={seed}  model={model_name} {'='*10}")
        t0 = time.time()
        p2, val_aucpr = train_one_seed_final(seed, ewt_s, ewt_r)
        test_probs = predict_traj_probs(p2, ewt_s, ewt_r, X2_t, starts2_t)
        y_test_agg = ytraj2.max(axis=-1).reshape(-1)
        test_agg_score = test_probs.max(axis=-1).reshape(-1)
        test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
        test_aucpr = average_precision_score(y_test_agg, test_agg_score)
        print(f"  trained in {time.time()-t0:.0f}s  val_AUCPR={val_aucpr:.4f}  "
              f"test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")
        results[model_name].append({"seed": seed, "val_aucpr": float(val_aucpr),
                                     "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr)})
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# SUMMARY + PAIRED SIGNIFICANCE TESTS
# ================================================================
wind_aucpr = np.array([r["test_aucpr"] for r in results["wind_graph"]])
nograph_aucpr = np.array([r["test_aucpr"] for r in results["no_graph"]])
wind_aucroc = np.array([r["test_aucroc"] for r in results["wind_graph"]])
nograph_aucroc = np.array([r["test_aucroc"] for r in results["no_graph"]])

print(f"\n{'='*20} SUMMARY ACROSS {N_SEEDS} SEEDS {'='*20}")
print(f"wind_graph  AUC-PR:  mean={wind_aucpr.mean():.4f}  std={wind_aucpr.std():.4f}  values={wind_aucpr.round(4)}")
print(f"no_graph    AUC-PR:  mean={nograph_aucpr.mean():.4f}  std={nograph_aucpr.std():.4f}  values={nograph_aucpr.round(4)}")
print(f"wind_graph  AUC-ROC: mean={wind_aucroc.mean():.4f}  std={wind_aucroc.std():.4f}  values={wind_aucroc.round(4)}")
print(f"no_graph    AUC-ROC: mean={nograph_aucroc.mean():.4f}  std={nograph_aucroc.std():.4f}  values={nograph_aucroc.round(4)}")

t_stat_pr, p_ttest_pr = stats.ttest_rel(wind_aucpr, nograph_aucpr)
w_stat_pr, p_wilcoxon_pr = stats.wilcoxon(wind_aucpr, nograph_aucpr)
t_stat_roc, p_ttest_roc = stats.ttest_rel(wind_aucroc, nograph_aucroc)
w_stat_roc, p_wilcoxon_roc = stats.wilcoxon(wind_aucroc, nograph_aucroc)

n_wins_pr = int((wind_aucpr > nograph_aucpr).sum())
n_wins_roc = int((wind_aucroc > nograph_aucroc).sum())

print(f"\nAUC-PR:  paired t-test t={t_stat_pr:.3f} p={p_ttest_pr:.4f}  |  Wilcoxon p={p_wilcoxon_pr:.4f}  |  wind_graph wins {n_wins_pr}/{N_SEEDS} seeds")
print(f"AUC-ROC: paired t-test t={t_stat_roc:.3f} p={p_ttest_roc:.4f}  |  Wilcoxon p={p_wilcoxon_roc:.4f}  |  wind_graph wins {n_wins_roc}/{N_SEEDS} seeds")

with open(f"{BASE}/wind_vs_nograph_10seed.json", "w") as f:
    json.dump({"wind_graph": results["wind_graph"], "no_graph": results["no_graph"],
               "aucpr_paired_ttest": {"t": float(t_stat_pr), "p": float(p_ttest_pr)},
               "aucpr_wilcoxon": {"stat": float(w_stat_pr), "p": float(p_wilcoxon_pr)},
               "aucroc_paired_ttest": {"t": float(t_stat_roc), "p": float(p_ttest_roc)},
               "aucroc_wilcoxon": {"stat": float(w_stat_roc), "p": float(p_wilcoxon_roc)}}, f, indent=2)
print(f"\nsaved to {BASE}/wind_vs_nograph_10seed.json")


device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17473 val=8689 test=8689
pos_weight (per-hour trajectory): 50.00

========== seed=0  model=wind_graph ==========
  trained in 653s  val_AUCPR=0.5023  test_AUCROC=0.9583  test_AUCPR=0.7157

========== seed=0  model=no_graph ==========
  trained in 643s  val_AUCPR=0.4604  test_AUCROC=0.9466  test_AUCPR=0.6752

========== seed=1  model=wind_graph ==========
  trained in 640s  val_AUCPR=0.4873  test_AUCROC=0.9572  test_AUCPR=0.7033

========== seed=1  model=no_graph ==========
  trained in 641s  val_AUCPR=0.4716  test_AUCROC=0.9504  test_AUCPR=0.6874

========== seed=2  model=wind_graph ==========
  trained in 640s  val_AUCPR=0.4894  test_AUCROC=0.9631  test_AUCPR=0.7193

========== seed=2  model=no_graph ==========
  trained in 645s  val_AUCPR=0.4693  test_AUCROC=0.9501  test_AUCPR=0.6889

========== seed=3  model=wind_graph ==========
  trained in 640s  val_AUCPR=0.5026  test_AUCROC=

In [ ]:
#obtain the advanced warning time of the graph method vs no graph method. 

In [ ]:
#get the final numbers for publication (auc, auc pr, recall etc.)

In [ ]:
#make plots, do exploratory analysis etc. 